<h1 style=\"text-align: center; font-size: 50px;\"> 📜 Text Generation with Neural Networks and Tensorflow</h1>

In this notebook our objective is to demonstrate how to generate text using a character-based RNN and Tensorflow working with a dataset of Shakespeare's  writing

Notebook Overview
- Start Execution
- Install and Import Libraries
- Configure Settings
- Verify Assets
- Get Text Data
- Preparing textual data
- Text Vectorization
- Creating Training Batches
- Creating the GRU Model
- Instance of the Model
- Training the model
- Saving the Model
- Load Model
- Generating Predictions

## Install and Import Libraries

In [1]:
%%time

%pip install -r ../../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.
CPU times: user 63 ms, sys: 27.7 ms, total: 90.7 ms
Wall time: 1.76 s


In [2]:
# -----------------------------
# Standard library imports
# -----------------------------
import logging              # Logging utilities
import os                   # Operating system utilities (paths, env vars, etc.)
import sys                  # Python runtime environment manipulation
import time                 # Time-related utilities
import warnings             # Warning control and message handling
from datetime import datetime  # Date and time handling
from pathlib import Path     # Object-oriented filesystem paths

# -----------------------------
# Third-party imports
# -----------------------------
import numpy as np           # Numerical computations and arrays
import tensorflow as tf      # TensorFlow deep learning framework
from tensorflow.keras.callbacks import TensorBoard  # Training visualization callback
from tensorflow.keras.layers import Dense, Embedding, GRU, InputLayer, LSTM  # Neural network layers
from tensorflow.keras.losses import sparse_categorical_crossentropy  # Loss function
from tensorflow.keras.models import Sequential, load_model  # Model definition and loading

import torch                 # PyTorch deep learning framework
from torch import nn          # Neural network layers and modules

# -----------------------------
# Local imports
# -----------------------------
# Extend sys.path to allow importing from parent directory
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

from src.utils import logger  # Project-specific logging utility

2025-12-01 18:14:59.966254: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-01 18:15:00.000103: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764612900.019785    2595 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764612900.025677    2595 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-01 18:15:00.053606: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

## Start Execution

In [3]:
start_time = time.time()  

logger.info("Notebook execution started.")

## Configure Settings

In [4]:
warnings.filterwarnings("ignore")

In [5]:
# Define global experiment and run names to be used throughout the notebook
MODEL_NAME = "tf_rnn_model.h5"

# Set up the paths
DATA_PATH = "../../data/shakespeare.txt"
TENSORBOARD_PATH = "/phoenix/tensorboard/tensorlogs"


# Set up the chunk separator for text processing
CHUNK_SEPARATOR = "\n\n"

## Verify Assets

In [6]:
def log_asset_status(asset_path: str, asset_name: str, success_message: str, failure_message: str) -> None:
    """
    Logs the status of a given asset based on its existence.

    Parameters:
        asset_path (str): File or directory path to check.
        asset_name (str): Name of the asset for logging context.
        success_message (str): Message to log if asset exists.
        failure_message (str): Message to log if asset does not exist.
    """
    if Path(asset_path).exists():
        logger.info(f"{asset_name} is properly configured. {success_message}")
    else:
        logger.error(f"{asset_name} is not properly configured. {failure_message}")
        
log_asset_status(
    asset_path=DATA_PATH,
    asset_name="Shakespeare text",
    success_message="",
    failure_message="Please download the required assets in your project on AI Studio."
)

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Get Text Data

This is the text we'll use as a basis for our generations: let's try to generate 'Shakespearean' texts.

This text is from Shakespeare's Sonnet 1. It's one of the 154 sonnets written by William Shakespeare that were first published in 1609. This particular sonnet, like many others, discusses themes of beauty, procreation, and the transient nature of life, urging the beautiful to reproduce so their beauty can live on through their offspring.

In [8]:
path_to_file = DATA_PATH
text = open(path_to_file, 'r').read()

In [9]:
logger.info('First 600 chars: \n')
print(text[:600])


                     1
  From fairest creatures we desire increase,
  That thereby beauty's rose might never die,
  But as the riper should by time decease,
  His tender heir might bear his memory:
  But thou contracted to thine own bright eyes,
  Feed'st thy light's flame with self-substantial fuel,
  Making a famine where abundance lies,
  Thy self thy foe, to thy sweet self too cruel:
  Thou that art now the world's fresh ornament,
  And only herald to the gaudy spring,
  Within thine own bud buriest thy content,
  And tender churl mak'st waste in niggarding:
    Pity the world, or else th


## Preparing textual data

We need to encode our data to give the model a proper numerical representation of our text.

In [10]:
# creates a set of unique characters found in the text
vocab = sorted(set(text))

## Text Vectorization

In [11]:
char_to_int = {u:i for i, u in enumerate(vocab)}
# assigns a unique integer to each character in a dictionary format, 
# creating a mapping that can later be used to transform encoded predictions back into characters

In [12]:
int_to_char = np.array(vocab)
# reverses the decoder dictionary, providing a mapping from characters to their respective assigned integers, which is used to encode the text.

In [13]:
encoded_text = np.array([char_to_int[c] for c in text])
# encodes the entire text as an array of integers, with each integer representing the character at that position
# in the text according to the encoder dictionary

## Creating Training Batches

Training batches are a way of dividing the dataset into smaller, manageable groups of data points that are fed into a machine learning model during the training process.

In [14]:
seq_len = 120 # length of sequence for a training example
total_num_seq = len(text)//(seq_len+1) # total number of training examples

# Create Training Sequences
char_dataset = tf.data.Dataset.from_tensor_slices(encoded_text)
sequences = char_dataset.batch(seq_len+1, drop_remainder=True)

I0000 00:00:1764612910.240761    2595 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 45687 MB memory:  -> device: 0, name: NVIDIA RTX 6000 Ada Generation, pci bus id: 0000:21:00.0, compute capability: 8.9


In [15]:
def create_seq_targets(seq):
    """
    Function that takes a sequence as input, duplicates, and shifts it to align the input and label. 

    Args:
        seq: sequence of characters

    Returns:
        The text input and corresponding target.
    """
    try:
        input_txt = seq[:-1]
        target_txt = seq[1:]
        return input_txt, target_txt
    except Exception as e:
            logger.error(f"Error creating sequences of targets: {str(e)}")

In [16]:
dataset = sequences.map(create_seq_targets)

In [17]:
# Batch size
batch_size = 128
buffer_size = 10000

dataset = dataset.shuffle(buffer_size).batch(batch_size, drop_remainder=True)

## Creating the GRU Model

In [18]:
# Length of the vocabulary in chars
vocab_size = len(vocab)
# The embedding dimension
embed_dim = 64
# Number of RNN units
rnn_neurons = 1026

In [19]:
def sparse_cat_loss(y_true,y_pred):
  return sparse_categorical_crossentropy(y_true, y_pred, from_logits=True)

In [20]:
def create_model(vocab_size, embed_dim, rnn_neurons, batch_size):
    """Architecture to create the model.

    Args:
        vocab_size: Length of the vocabulary in chars.
        embed_dim: Embedding dimension.
        rnn_neurons: Number of RNN units.
        batch_size: Size of the batchs.

    Returns:
        Model.
    """
    try:
        model = Sequential()
        model.add(InputLayer(batch_shape=(batch_size, None)))
        
        model.add(Embedding(input_dim=vocab_size, output_dim=embed_dim))

        model.add(GRU(rnn_neurons,
                    return_sequences=True,
                    stateful=True,
                    recurrent_initializer='glorot_uniform'))

        model.add(Dense(vocab_size))
        model.compile(optimizer='adam', loss=sparse_cat_loss)
        logger.info("Model architecture created successfully")
        return model
    except Exception as e:
            logger.error(f"Error creating model architecture: {str(e)}")

model = create_model(vocab_size, embed_dim, rnn_neurons, batch_size)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (128, None, 64)        │         5,376 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (128, None, 1026)      │     3,361,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (128, None, 84)        │        86,268 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,452,820 (13.17 MB)

 Trainable params: 3,452,820 (13.17 MB)

 Non-trainable params: 0 (0.00 B)

## Instance of the Model

In [21]:
model = create_model(
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    rnn_neurons=rnn_neurons,
    batch_size=batch_size
)


In [22]:
# TensorBoard
log_dir = TENSORBOARD_PATH
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

## Training the model

In [23]:
for input_example_batch, target_example_batch in dataset.take(1):

  # Predict off some random batch
  example_batch_predictions = model(input_example_batch)

  # Display the dimensions of the predictions
  print(example_batch_predictions.shape, " <=== (batch_size, sequence_length, vocab_size)")

I0000 00:00:1764612922.053581    2712 cuda_dnn.cc:529] Loaded cuDNN version 91002


(128, 120, 84)  <=== (batch_size, sequence_length, vocab_size)


2025-12-01 18:15:22.298949: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [24]:
sampled_indices = tf.random.categorical(example_batch_predictions[0], num_samples=1)
# Reformat to not be a lists of lists
sampled_indices = tf.squeeze(sampled_indices,axis=-1).numpy()

In [25]:
%%time

epochs = 20
model.fit(dataset,epochs=epochs, callbacks=[tensorboard_callback])

Epoch 1/20
350/350 ━━━━━━━━━━━━━━━━━━━━ 12s 26ms/step - loss: 2.2895
Epoch 2/20
350/350 ━━━━━━━━━━━━━━━━━━━━ 10s 24ms/step - loss: 1.5658
Epoch 3/20
350/350 ━━━━━━━━━━━━━━━━━━━━ 10s 24ms/step - loss: 1.3716
Epoch 4/20
350/350 ━━━━━━━━━━━━━━━━━━━━ 10s 24ms/step - loss: 1.2891
Epoch 5/20
350/350 ━━━━━━━━━━━━━━━━━━━━ 10s 25ms/step - loss: 1.2424
Epoch 6/20
350/350 ━━━━━━━━━━━━━━━━━━━━ 10s 24ms/step - loss: 1.2101
Epoch 7/20
350/350 ━━━━━━━━━━━━━━━━━━━━ 10s 24ms/step - loss: 1.1848
Epoch 8/20
350/350 ━━━━━━━━━━━━━━━━━━━━ 10s 26ms/step - loss: 1.1652
Epoch 9/20
350/350 ━━━━━━━━━━━━━━━━━━━━ 14s 36ms/step - loss: 1.1474
Epoch 10/20
350/350 ━━━━━━━━━━━━━━━━━━━━ 10s 24ms/step - loss: 1.1326
Epoch 11/20
350/350 ━━━━━━━━━━━━━━━━━━━━ 10s 24ms/step - loss: 1.1184
Epoch 12/20
350/350 ━━━━━━━━━━━━━━━━━━━━ 10s 24ms/step - loss: 1.1065
Epoch 13/20
350/350 ━━━━━━━━━━━━━━━━━━━━ 10s 24ms/step - loss: 1.0946
Epoch 14/20
350/350 ━━━━━━━━━━━━━━━━━━━━ 10s 24ms/step - loss: 1.0839
Epoch 15/20
350/350 ━━━━━━━━━

## Saving the Model

In [26]:
model_name = MODEL_NAME

In [27]:
model.save(f'models/{model_name}') 
logger.info("Model saved")

INFO:AIS_logger:Model saved


## Load Model

In [28]:
model = create_model(vocab_size, embed_dim, rnn_neurons, batch_size=1)

model.load_weights(f'models/{model_name}')

model.build(tf.TensorShape([1, None]))

INFO:AIS_logger:Model architecture created successfully


# Generating Predictions

In [29]:
def generate_text(model, start_seed="The ", gen_size=100, temp=1.0):
    """
    Generates a sequence of text using the trained character-level language model.

    Args:
        model: Model created on function create_model
        start_seed: Set of characters that will be the beginning of the text. 
        gen_size : Number of characters. Defaults to 100.
        temp: Controls the randomness of the predictions made by the model.

    Returns:
        The full generated text including the seed and the newly predicted characters.
    """
    try:
        num_generate = gen_size
        input_eval = [char_to_int[s] for s in start_seed]
        input_eval = tf.expand_dims(input_eval, 0)
        text_generated = []
        temperature = temp


        for i in range(num_generate):
            predictions = model(input_eval)
            predictions = tf.squeeze(predictions, 0)
            predictions = predictions / temperature
            predicted_id = tf.random.categorical(predictions, num_samples=1)[-1,0].numpy()
            input_eval = tf.expand_dims([predicted_id], 0)
            text_generated.append(int_to_char[predicted_id])

        return start_seed + ''.join(text_generated)
    except Exception as e:
            logger.error(f"Error making predictions: {str(e)}")


#### Generating a text with 1000 chars starting with word 'Confidence'

In [30]:
for layer in model.layers:
    if hasattr(layer, 'reset_states'):
        layer.reset_states()
print(generate_text(model, start_seed="Confidence", gen_size=1000))

Confidence
    That knits your brother's princely very enemy
    With other blother, and of these five lips of the rs,
    Within this hour, here comes thy lunce for thy master.
  JULIA. O my deep waste here! Who goes to do his wife,
    How shall I hear all   And the best joins in love bears nothing rank
    Than thou art thrown.
    Then in her day never come
    Imperious it at our Calliard the stream that mov'd it
    Will keep me in my thronichable rescu'd,
    But with thy keep my state, and take her love,
    Or then have merited to the pheeping sharp
    Upon this gentleman he hath beaten this
    head and say he cannot choose to-night.
  ANTIGONUS. I'll begin, my giving of, sir; but I thank thee
    That do as suit as thinking of whom here
    Show'red joints in death with love. You shall have the matter
    What I have wound a tall. I should have measur'd
    With strifical pluntmy stockings gloss of soul
    And say they have their horse tonight to be our proceeding safesy,


#### Generating a text with 1000 chars starting with word 'Love'

In [31]:
print(generate_text(model, start_seed="Love", gen_size=1000))

Lovel God and henceless three or form,
    And bamble as too welk what evils must be to to their anchors,
  That norlest her foreglory without gave me their
  Than my affairs,
     With costly fristy bodies to my hands.
  TRANIO. My lord, you hear the heirland too language.'
    And so do I perceitest except thee,
    Unbaded as it were as true to thee,
    Some state I am out in your broken feet.
    Thou canst not suffer her.
  OLIVIA. 'Sir- because it were as soul as yours?
  DESDEMONA. Say that you call'd us so like's sons commanded
    Have my money you by this foul brings
    Which to his chin. If she did help'd out of fault's
    turn'd brothers; and told me the retire and sight on a hair standly hath done,
    And she shall seems have verched too,
    Unkingn and infancy and weak design,
    And thy Achilles married to an under vulgurous
    this have crown'd him. I will torment my countenance and the old lad prey. My lady's hook
    and odd sign of white weeks expectation agai

In [32]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")
logger.info("✅ Notebook execution completed successfully.")

INFO:AIS_logger:⏱️ Total execution time: 4m 25.72s


INFO:AIS_logger:✅ Notebook execution completed successfully.


Built with ❤️ using [**HP AI Studio**](https://hp.com/ai-studio).